# 📊 Understanding Your Data
### D4BL Tutorial 1 of 5

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/William-Hill/d4bl_ai_agent/blob/main/notebooks/tutorials/01_understanding_your_data.ipynb)

**What you'll learn:** How equity data is structured, what disparities look like in numbers, and how D4BL organizes data from 17 federal and state sources.

**Time:** ~15 minutes | **Prerequisites:** A Google account (for Colab) | **Dependencies:** pandas, matplotlib (pre-installed)

## What is Equity Data?

Data for Black Lives works with data from 17 sources including the CDC, U.S. Census Bureau, EPA, FBI, and more. This data tracks health outcomes, economic indicators, environmental hazards, and criminal justice metrics — broken down by race, geography, and time.

**Why it matters:** Generic datasets often bury racial disparities in averages. When you break data out by race and place, patterns emerge that reflect decades of structural decisions — redlining, disinvestment, over-policing. Understanding the shape of this data is the first step toward building AI that sees these patterns instead of ignoring them.

D4BL's data sources include:

| Source | What It Covers | Example Metrics |
|--------|---------------|------------------|
| CDC PLACES | Community health | Diabetes prevalence, mental health, preventive care |
| Census ACS | Economic indicators | Median income, poverty rate, homeownership |
| EPA EJScreen | Environmental justice | PM2.5, lead paint, proximity to hazardous waste |
| FBI UCR | Criminal justice | Arrest rates, offense types |
| BLS | Employment | Unemployment rate by race |

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# D4BL brand colors
D4BL_GREEN = "#00ff32"
D4BL_BG = "#292929"
D4BL_TEXT = "#e5e5e5"

## Exploring Sample Data

Below is a small sample from three of D4BL's data sources. In the real system, this data lives in a PostgreSQL database with over 25 tables. Here we'll work with embedded samples to understand the structure.

In [ ]:
# Sample CDC PLACES data — community health metrics by state and race
cdc_data = [
    {"state": "AL", "metric": "diabetes_prevalence", "race": "black", "value": 16.8, "year": 2022},
    {"state": "AL", "metric": "diabetes_prevalence", "race": "white", "value": 11.2, "year": 2022},
    {"state": "GA", "metric": "diabetes_prevalence", "race": "black", "value": 15.9, "year": 2022},
    {"state": "GA", "metric": "diabetes_prevalence", "race": "white", "value": 10.4, "year": 2022},
    {"state": "MS", "metric": "diabetes_prevalence", "race": "black", "value": 18.1, "year": 2022},
    {"state": "MS", "metric": "diabetes_prevalence", "race": "white", "value": 12.7, "year": 2022},
    {"state": "AL", "metric": "mental_health_not_good", "race": "black", "value": 19.2, "year": 2022},
    {"state": "AL", "metric": "mental_health_not_good", "race": "white", "value": 16.8, "year": 2022},
]

cdc_df = pd.DataFrame(cdc_data)
print("CDC PLACES sample:")
cdc_df

In [ ]:
# Sample Census ACS data — economic indicators
census_data = [
    {"state": "AL", "metric": "median_household_income", "race": "black", "value": 35210, "year": 2022},
    {"state": "AL", "metric": "median_household_income", "race": "white", "value": 58640, "year": 2022},
    {"state": "AL", "metric": "poverty_rate", "race": "black", "value": 28.3, "year": 2022},
    {"state": "AL", "metric": "poverty_rate", "race": "white", "value": 11.5, "year": 2022},
    {"state": "GA", "metric": "median_household_income", "race": "black", "value": 42180, "year": 2022},
    {"state": "GA", "metric": "median_household_income", "race": "white", "value": 72340, "year": 2022},
    {"state": "GA", "metric": "poverty_rate", "race": "black", "value": 22.1, "year": 2022},
    {"state": "GA", "metric": "poverty_rate", "race": "white", "value": 9.8, "year": 2022},
]

census_df = pd.DataFrame(census_data)
print("Census ACS sample:")
census_df

In [ ]:
# Sample EPA EJScreen data — environmental justice indicators
epa_data = [
    {"state": "AL", "metric": "PM25", "percentile_minority": 78, "percentile_lowinc": 72, "year": 2023},
    {"state": "AL", "metric": "lead_paint", "percentile_minority": 85, "percentile_lowinc": 80, "year": 2023},
    {"state": "GA", "metric": "PM25", "percentile_minority": 71, "percentile_lowinc": 65, "year": 2023},
    {"state": "MS", "metric": "PM25", "percentile_minority": 82, "percentile_lowinc": 79, "year": 2023},
]

epa_df = pd.DataFrame(epa_data)
print("EPA EJScreen sample:")
epa_df

## Spotting Disparities

The most basic equity analysis: compare outcomes across racial groups. A **disparity ratio** tells you how many times worse the outcome is for one group compared to another. A ratio of 2.0 means twice as bad.

These ratios aren't random — they reflect structural conditions: which neighborhoods got investment, which communities got environmental hazards, who got access to healthcare.

In [ ]:
# Compute Black/white disparity ratios for each metric and state
def compute_disparity(df, metric_name):
    """Compute the Black/white ratio for a given metric."""
    metric_df = df[df["metric"] == metric_name]
    black = metric_df[metric_df["race"] == "black"].set_index("state")["value"]
    white = metric_df[metric_df["race"] == "white"].set_index("state")["value"]
    ratio = black / white
    return ratio.round(2)

# Diabetes prevalence disparity
diabetes_ratio = compute_disparity(cdc_df, "diabetes_prevalence")
print("Diabetes prevalence — Black/white ratio by state:")
print(diabetes_ratio)
print()

# Poverty rate disparity
poverty_ratio = compute_disparity(census_df, "poverty_rate")
print("Poverty rate — Black/white ratio by state:")
print(poverty_ratio)

In [ ]:
# Visualize the income gap
income_df = census_df[census_df["metric"] == "median_household_income"]

fig, ax = plt.subplots(figsize=(8, 4))
fig.patch.set_facecolor(D4BL_BG)
ax.set_facecolor(D4BL_BG)

states = income_df["state"].unique()
x = range(len(states))
width = 0.35

black_vals = [income_df[(income_df["state"] == s) & (income_df["race"] == "black")]["value"].iloc[0] for s in states]
white_vals = [income_df[(income_df["state"] == s) & (income_df["race"] == "white")]["value"].iloc[0] for s in states]

ax.bar([i - width/2 for i in x], black_vals, width, label="Black", color=D4BL_GREEN)
ax.bar([i + width/2 for i in x], white_vals, width, label="White", color="#666666")
ax.set_xticks(x)
ax.set_xticklabels(states, color=D4BL_TEXT)
ax.set_ylabel("Median Household Income ($)", color=D4BL_TEXT)
ax.set_title("Income Gap by State", color=D4BL_TEXT, fontsize=14)
ax.legend()
ax.tick_params(colors=D4BL_TEXT)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"${x:,.0f}"))
plt.tight_layout()
plt.show()

## Data Shape for Training

When we fine-tune a language model, we don't feed it raw numbers. We convert data rows into natural language that the model can learn from. Here's a preview of how a Census ACS row becomes a training input:

**Raw data row:**
```
{"state": "AL", "metric": "poverty_rate", "race": "black", "value": 28.3}
```

**As a training prompt:**
> "Explain the poverty rate disparity in Alabama, where the Black poverty rate is 28.3% compared to 11.5% for white residents."

The model learns to respond with equity-framed analysis — naming structural causes, connecting to policy, and acknowledging data limitations. That's what Notebook 2 covers.

## ✏️ Exercise

Pick a different metric from the sample data above (e.g., `mental_health_not_good` from CDC, or `median_household_income` from Census) and:

1. Compute the Black/white disparity ratio
2. Create a bar chart comparing the groups
3. Write one sentence explaining what structural factor might contribute to this gap

In [ ]:
# TODO: Pick a metric and compute the disparity ratio
# metric_name = "..."
# ratio = compute_disparity(??_df, metric_name)
# print(ratio)

## Summary

You've seen how D4BL organizes equity data from 17 sources, computed disparity ratios, and visualized the income gap. These numbers aren't abstract — they reflect structural conditions that policy can change.

**Next:** [Notebook 2 — Creating Training Data](https://colab.research.google.com/github/William-Hill/d4bl_ai_agent/blob/main/notebooks/tutorials/02_creating_training_data.ipynb) → Learn how this data becomes training examples for a fine-tuned model.